# Concepts Notebook — Momentum Game-Theoretic Strategy
**Hugh Hayes · QUANTT**

The core probability, statistics, and financial concepts the strategy uses.
Each concept has: a short explanation, a simple function, and a worked example.

This is a reference for what the strategy is built on, not the strategy itself.
(Not all will be used, for example 12-1 correlation will be switched to 36 prob).


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)  # reproducible examples

## 1. Returns (simple & log)

A **return** is the percentage change in price.

- **Simple return:** `(P_today / P_yesterday) - 1`. Intuitive; what you actually earn.
- **Log return:** `ln(P_today / P_yesterday)`. Adds up nicely over time, used in some stats. I like for large incremental calculations, smoother.

The strategy uses **simple returns** for performance and **log returns** occasionally for
statistical work. For small moves they're nearly identical.

In [3]:
def simple_return(p_now, p_prev):
    return (p_now / p_prev) - 1

def log_return(p_now, p_prev):
    return np.log(p_now / p_prev)

# Example: stock goes from $100 to $105
print("Simple return:", simple_return(105, 100))   # 0.05  = +5%
print("Log return:   ", round(log_return(105, 100), 5))  # 0.04879

Simple return: 0.050000000000000044
Log return:    0.04879


## 2. Cumulative Return & Equity Curve

The **equity curve** tracks how $1 grows as returns compound period after period.
You multiply (1 + return) across all periods. you don't add them, because each period
compounds on the previous balance.

**Cumulative return** = final equity value − 1.

In [4]:
def equity_curve(returns):
    return (1 + pd.Series(returns)).cumprod()

def cumulative_return(returns):
    return equity_curve(returns).iloc[-1] - 1

# Example: three months ofÍ +2%, -1%, +3%
r = [0.02, -0.01, 0.03]
print("Equity curve:")
print(equity_curve(r).round(4).tolist())   # [1.02, 1.0098, 1.0401]
print("Cumulative return:", round(cumulative_return(r), 4))  # 0.0401 = +4.01%

Equity curve:
[1.02, 1.0098, 1.0401]
Cumulative return: 0.0401


## 3. Annualization

Backtests span many months, so you convert to a per-year figure for comparison.

- **Annualized return:** `final_equity ^ (12 / n_months) - 1` (geometric, accounts for compounding)
- **Annualized volatility:** `monthly_std * sqrt(12)` (volatility scales with √time, not linearly)

Use 12 for monthly data, 252 for daily.

In [5]:
def annualized_return(returns, periods_per_year=12):
    eq = (1 + pd.Series(returns)).prod()
    n = len(returns)
    return eq ** (periods_per_year / n) - 1

def annualized_vol(returns, periods_per_year=12):
    return np.std(returns, ddof=1) * np.sqrt(periods_per_year)

# Example: 60 monthly returns averaging ~0.5%/month
monthly = np.random.normal(0.005, 0.02, 60)
print("Annualized return:", round(annualized_return(monthly), 4))
print("Annualized vol:   ", round(annualized_vol(monthly), 4))

Annualized return: 0.0211
Annualized vol:    0.0629


## 4. Dollar-Neutral Long/Short

The strategy goes **long** winners and **short** losers with equal dollar amounts on each side,
so the weights sum to 0. This means **net market exposure = 0** — if the whole market drops,
your longs lose but your shorts gain, roughly cancelling. You profit from the *spread* between
winners and losers, not market direction.

In [5]:
def dollar_neutral_weights(longs, shorts):
    w = {}
    for t in longs:
        w[t] = +0.5 / len(longs)    # +50% spread across longs
    for t in shorts:
        w[t] = -0.5 / len(shorts)   # -50% spread across shorts
    return w

# Example: 2 longs, 2 shorts
w = dollar_neutral_weights(["AAPL", "MSFT"], ["XYZ", "ABC"])
print(w)
print("Sum of weights:", round(sum(w.values()), 10))  # ~0.0 -> dollar neutral

{'AAPL': 0.25, 'MSFT': 0.25, 'XYZ': -0.25, 'ABC': -0.25}
Sum of weights: 0.0


## 5. Cross-Sectional Momentum (12-1)

The raw momentum signal we build *on top of*. For each stock, measure its return over the past 12 months **excluding the most recent month** (the "12-1" or "12-minus-1" window). Skipping the last month avoids **short-term reversal**, where last month's movers tend to reverse.

"Cross-sectional" = rank stocks against *each other*, then long the top, short the bottom.

In the final strategy we don't rank on this raw number directly. we first strip out market and factor noise (Sections 6-7) and rank on **residual** momentum instead. This section is the foundation that idea refines.

In [6]:
def momentum_12_1(price_series):

    
    p_1mo_ago  = price_series.iloc[-21]    #1 month ago (21 trading days)
    p_12mo_ago = price_series.iloc[-252]   #12 months ago
    return (p_1mo_ago / p_12mo_ago) - 1

# Example: build a fake 1-year price path that rose then dipped recently
dates = pd.bdate_range("2023-01-01", periods=260)
prices = pd.Series(np.linspace(80, 110, 260), index=dates)  # rose 80 -> 110
prices.iloc[-21:] = np.linspace(110, 105, 21)               # dipped last month
print("12-1 momentum signal:", round(momentum_12_1(prices), 4))

# Cross-sectional ranking across several stocks:
signals = pd.Series({"AAPL": 0.30, "MSFT": 0.22, "XOM": -0.05,
                     "GE": -0.18, "F": 0.11})


ranked = signals.sort_values(ascending=False)
print("\nRanked by momentum:")
print(ranked)
print("Longs (top 2): ", list(ranked.head(2).index))
print("Shorts (bot 2):", list(ranked.tail(2).index))

12-1 momentum signal: 0.3593

Ranked by momentum:
AAPL    0.30
MSFT    0.22
F       0.11
XOM    -0.05
GE     -0.18
dtype: float64
Longs (top 2):  ['AAPL', 'MSFT']
Shorts (bot 2): ['XOM', 'GE']


## 6. Factor Models & OLS Regression

A **factor model** explains a stock's return as the part driven by common factors plus a stock-specific leftover:

`stock_return = alpha + b1*Mkt + b2*SMB + b3*HML + residual`

- **Mkt, SMB, HML** are the Fama-French factors (market, size, value) - downloaded free from Ken French's library.
- **alpha, b1, b2, b3** are the *betas*: this stock's sensitivity to each factor. We don't look them up. an OLS regression calculates them from the data.
- **residual** is what the factors can't explain: the stock's idiosyncratic return. This is what residual momentum (Section 7) ranks on.

The regression finds the betas that best fit the stock's history, then the residual is `actual - predicted`.

In [6]:
def factor_regression(stock_excess, factors):
    # stock_excess: Series of stock returns minus risk-free (length T)
    # factors: DataFrame with columns [Mkt, SMB, HML] (T rows)
    
    X = factors[["Mkt", "SMB", "HML"]].values
    X = np.column_stack([np.ones(len(X)), X])   # add intercept column for alpha
    y = stock_excess.values

    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)

    predicted = X @ coeffs
    residuals = y - predicted
    return coeffs, residuals   # coeffs = [alpha, b_mkt, b_smb, b_hml]

rng = np.random.default_rng(1)
factors = pd.DataFrame({
    "Mkt": rng.normal(0.005, 0.04, 36),
    "SMB": rng.normal(0.001, 0.02, 36),
    "HML": rng.normal(0.000, 0.02, 36),
})

stock = 0.002 + 1.3*factors["Mkt"] + 0.4*factors["SMB"] + rng.normal(0, 0.01, 36)

coeffs, resids = factor_regression(stock, factors)
print("alpha, b_mkt, b_smb, b_hml:", np.round(coeffs, 3))
print("First 3 residuals:        ", np.round(resids[:3], 4))

alpha, b_mkt, b_smb, b_hml: [ 0.     1.36   0.41  -0.055]
First 3 residuals:         [ 0.0173 -0.0006 -0.0061]


## 7. Residual Momentum (the core signal)

The actual ranking signal of the strategy. Instead of ranking on raw 12-1 returns, we rank on **residual** momentum: momentum measured on the factor-model residuals from Section 6.

Steps for each stock:
1. Run a rolling 36-month factor regression. keep the residuals (idiosyncratic returns).
2. Sum the residuals over the 12-1 window (months t-12 to t-1, skipping the most recent month).
3. **Standardize** by dividing by the standard deviation of those residuals.

This strips out market, size, and value moves so we rank on genuine stock-specific momentum. 

In [15]:
def residual_momentum(residuals, skip=1):
    # residuals: Series of monthly idiosyncratic returns (most recent last)
    # skip: how many recent months to skip (1 = standard 12-1)
    window = residuals.iloc[-(12 + skip):-skip] if skip else residuals.iloc[-12:]
    return window.sum() / window.std()   # standardized residual momentum

# Example: 13 months of residuals - positive idiosyncratic drift with realistic noise
rng = np.random.default_rng(7)
res = pd.Series(rng.normal(0.008, 0.02, 12).tolist() + [-0.04])
# last value (-0.04) is the most recent month and gets skipped
print("Residual momentum score:", round(residual_momentum(res), 3))

Residual momentum score: 5.47


## 8. Volatility & Standard Deviation

**Volatility** is the standard deviation of returns — how much they bounce around the average.
High volatility = bumpy/risky. It's the denominator of the Sharpe ratio and the input to
position sizing.

In [8]:
def volatility(returns):
    return np.std(returns, ddof=1)   # ddof=1 = sample std

# Example: two return streams, same average, different bumpiness
calm   = [0.01, 0.012, 0.009, 0.011, 0.008]
wild   = [0.05, -0.03, 0.04, -0.02, 0.03]
print("Calm vol:", round(volatility(calm), 4))
print("Wild vol:", round(volatility(wild), 4))

Calm vol: 0.0016
Wild vol: 0.0365


## 9. Sharpe Ratio

The headline risk-adjusted return metric: **return per unit of risk**.

Sharpe = (annualized return - risk-free rate) / annualized volatility



In [9]:
def sharpe_ratio(returns, rf_annual=0.0, periods_per_year=12):
    ann_ret = annualized_return(returns, periods_per_year)
    ann_vol = annualized_vol(returns, periods_per_year)
    return (ann_ret - rf_annual) / ann_vol

# Example
monthly = np.random.normal(0.008, 0.02, 60)
print("Sharpe ratio:", round(sharpe_ratio(monthly, rf_annual=0.03), 3))

Sharpe ratio: 1.027


## 10. Sortino Ratio

Like Sharpe, but only penalizes **downside** volatility (losses), not upside swings.
Uses the standard deviation of *negative* returns in the denominator. Rewards strategies
whose volatility is mostly to the upside. I think this is a better metric than sharpe as momentum has high upside volatile swings.

In [10]:
def sortino_ratio(returns, rf_annual=0.0, periods_per_year=12):
    ann_ret = annualized_return(returns, periods_per_year)
    downside = [r for r in returns if r < 0]
    downside_vol = np.std(downside, ddof=1) * np.sqrt(periods_per_year)
    return (ann_ret - rf_annual) / downside_vol

monthly = np.random.normal(0.008, 0.02, 60)
print("Sortino ratio:", round(sortino_ratio(monthly, rf_annual=0.03), 3))

Sortino ratio: 4.306


## 11. Maximum Drawdown

The worst **peak-to-trough** drop in the equity curve — your most painful losing stretch.
Measures how much you'd have lost if you bought at the worst possible peak. A key risk metric
because investors care about the deepest hole, not just average volatility.

In [11]:
def max_drawdown(returns):
    eq = (1 + pd.Series(returns)).cumprod()
    running_peak = eq.cummax()
    drawdown = (eq - running_peak) / running_peak
    return drawdown.min()   # most negative value

# Example: a run with a clear drop in the middle

r = [0.05, 0.03, -0.10, -0.08, 0.04, 0.06]
print("Max drawdown:", round(max_drawdown(r), 4))  

Max drawdown: -0.172


## 12. Correlation

Measures how two return streams move together: +1 = perfectly together, 0 = unrelated,
-1 = perfectly opposite. The strategy uses **pairwise correlations** among momentum stocks
to detect crowding (next section).

In [12]:
def correlation(returns_a, returns_b):
    return np.corrcoef(returns_a, returns_b)[0, 1]

a = [0.01, 0.02, -0.01, 0.03, -0.02]
b = [0.012, 0.018, -0.008, 0.025, -0.022]  # moves with a
c = [-0.01, -0.02, 0.01, -0.03, 0.02]      # moves against a
print("corr(a, b):", round(correlation(a, b), 3))   # near +1
print("corr(a, c):", round(correlation(a, c), 3))   # near -1

corr(a, b): 0.991
corr(a, c): -1.0


## 13. Z-Score / Standardization

Expresses a value as "how many standard deviations from the mean."
`z = (x - mean) / std`. Used to compare things on different scales and to detect extremes
(a z-score of +2 means unusually high). The comomentum signal is z-scored over time.

In [13]:
def zscore(value, history):
    return (value - np.mean(history)) / np.std(history, ddof=1)

# Example: is today's reading unusually high vs its history?
history = [0.30, 0.32, 0.28, 0.31, 0.29, 0.33]
today = 0.45
print("Z-score:", round(zscore(today, history), 2))  # large positive = unusual

Z-score: 7.75


## 14. Comomentum (Crowding Signal)

The distinctive piece of the strategy. **Comomentum** = the average pairwise correlation among the momentum stocks, measured on **market-residualized** returns. We first strip out the market move from each stock (so we don't count "everyone rose because the market rose" as crowding), then correlate what's left.

When the residual returns of momentum names move together, the trade is **crowded**: lots of capital chasing the same idiosyncratic signal, which historically precedes momentum crashes. You z-score comomentum over time; a high z-score = crowded = reduce exposure.

In [14]:
def comomentum(returns_matrix, market):
    # returns_matrix: DataFrame, columns = momentum stocks, rows = daily returns
    # market: Series of market returns (same index) - stripped out first

    # 1. residualize each stock against the market (remove common market move)
    resid = pd.DataFrame(index=returns_matrix.index)
    X = np.column_stack([np.ones(len(market)), market.values])
    for col in returns_matrix.columns:
        y = returns_matrix[col].values
        coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid[col] = y - X @ coeffs

    # 2. average off-diagonal correlation of the residuals
    corr = resid.corr()
    n = corr.shape[0]
    off_diag_sum = corr.values.sum() - n     # subtract diagonal (all 1s)
    return off_diag_sum / (n * (n - 1))

# Example: 4 momentum stocks sharing a market move + idiosyncratic noise
rng = np.random.default_rng(0)
market = pd.Series(rng.normal(0, 0.01, 100))
mat = pd.DataFrame({f"stk{i}": 1.1*market + rng.normal(0, 0.005, 100)
                    for i in range(4)})
print("Comomentum (residualized avg pairwise corr):", round(comomentum(mat, market), 3))

Comomentum (residualized avg pairwise corr): -0.025


## 15. Inverse-Volatility Position Sizing

Instead of equal dollars per stock, size positions by **1/volatility** so each contributes
similar *risk*. Calmer stocks get bigger positions, wilder stocks get smaller ones. This is
the volatility-scaling layer of the strategy.

In [16]:
def inverse_vol_weights(vols):
    inv = {t: 1.0 / v for t, v in vols.items()}
    total = sum(inv.values())
    return {t: w / total for t, w in inv.items()}

vols = {"CALM": 0.10, "MEDIUM": 0.20, "WILD": 0.40}
w = inverse_vol_weights(vols)
print({t: round(x, 3) for t, x in w.items()})
print("Sum:", round(sum(w.values()), 6))

{'CALM': 0.571, 'MEDIUM': 0.286, 'WILD': 0.143}
Sum: 1.0


---
## Summary

| Concept | Used for |
|---|---|
| Returns, cumulative, annualization | Turning prices into performance numbers |
| Dollar-neutral long/short | Portfolio construction (net market = 0) |
| Cross-sectional momentum 12-1 | The raw momentum building block |
| Factor models & residual momentum | The core signal: stock-specific momentum |
| Volatility, Sharpe, Sortino, drawdown | Risk & risk-adjusted performance |
| Correlation, z-score, comomentum | The crowding / momentum-crash filter |
| Inverse-vol sizing | Position sizing (volatility scaling) |

These are the building blocks. The strategy combines them: rank by **residual momentum** (factor-adjusted), filter by **comomentum** crowding, and size by **inverse volatility**.
